# Introduction to Agents — the three building blocks

An "AI agent" is just three things glued together:

1. **LLM call** — we send messages, we get a reply.
2. **Memory** — we keep the past messages so the model can refer back.
3. **Tools** — we let the model call our Python functions to do things it can't do on its own.

This notebook builds those three pieces from scratch, one at a time.

## Cell 1 — Pick a provider from `.env` and build a client

We support four providers behind the same OpenAI-style SDK:
- `openai` — hosted OpenAI
- `azure` — Azure OpenAI
- `ollama` — a local model server (OpenAI-compatible)
- `openrouter` — OpenRouter's OpenAI-compatible API

We read `MODEL_PROVIDER` from `.env` and produce two things downstream cells will use: a `client` and a `MODEL` name.

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI, AzureOpenAI

load_dotenv(find_dotenv())

provider = "openrouter"

if provider == "openai":
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

elif provider == "azure":
    client = AzureOpenAI(
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    )
    MODEL = os.getenv("AZURE_OPENAI_DEPLOYMENT")

elif provider == "ollama":
    client = OpenAI(
        base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"),
        api_key=os.getenv("OLLAMA_API_KEY", "ollama"),
    )
    MODEL = os.getenv("OLLAMA_MODEL")

elif provider == "openrouter":
    client = OpenAI(
        base_url=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
        api_key=os.getenv("OPENROUTER_API_KEY") or os.getenv("OPENROUTER_API"),
    )
    MODEL = os.getenv("OPENROUTER_MODEL")

else:
    raise ValueError(f"Unknown MODEL_PROVIDER: {provider!r}. Use 'openai', 'azure', 'ollama', or 'openrouter'.")

print(f"Provider: {provider}")
print(f"Model:    {MODEL}")

# The simplest possible LLM call

In [ ]:
system_prompt = "You are a friendly tutor. Answer in one or two short sentences."
user_prompt = "In plain English, what is a Large Language Model?"

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ],
)

print(response.choices[0].message.content)

In [ ]:
system_prompt = "You are a friendly tutor. Answer in one or two short sentences."
user_prompt = "What was my last question?"

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ],
)

print(response.choices[0].message.content)

In [ ]:
dict(response)

# Add memory (conversation history)

In [ ]:
import json

def chat(user_message: str) -> str:
    history.append({"role": "user", "content": user_message})
    response = client.chat.completions.create(model=MODEL, messages=history)
    reply = response.choices[0].message.content
    history.append({"role": "assistant", "content": reply})
    return reply

history = [
    {"role": "system", "content": "You are a friendly tutor. Keep replies short."}
]


print("Q1:", chat("Hi! My name is Artur and I like trains."))

print("\n--- raw history (this is the 'memory') ---")
print(json.dumps(history, indent=2, ensure_ascii=False))

print("============ SECOND CALL ===============")
print("Q2:", chat("What's my name and what do I enjoy?"))

print("\n--- raw history (this is the 'memory') ---")
print(json.dumps(history, indent=2, ensure_ascii=False))

# Add a tool: the LLM + Memory + Tools triad

In [ ]:
import random

# --- 1. The tool itself ---------------------------------------------------
JOKES = {
    "AI": [
        "Why did the neural network go to therapy? Too many unresolved layers.",
        "I asked an LLM for a joke. It hallucinated a punchline.",
        "My chatbot broke up with me. It said I was too predictable.",
        "How many prompt engineers does it take to change a bulb? Just one, but the prompt is 4 pages long.",
        "AGI is always five years away — and so is my deadline.",
    ],
    "programmer": [
        "There are 10 kinds of people: those who understand binary and those who don't.",
        "A SQL query walks into a bar, sees two tables, and asks: 'Mind if I join you?'",
        "Why do programmers prefer dark mode? Because light attracts bugs.",
        "It's not a bug, it's an undocumented feature.",
        "I would tell you a UDP joke, but you might not get it.",
    ],
    "animals": [
        "What do you call a fish wearing a crown? Your royal haddock.",
        "Why don't oysters share? Because they're shellfish.",
        "What do you call a sleeping bull? A bulldozer.",
        "How do you organize a space party for cats? You planet.",
        "Why did the chicken join a band? Because it had the drumsticks.",
    ],
    "humans": [
        "I told my wife she was drawing her eyebrows too high. She looked surprised.",
        "Parallel lines have so much in common — it's a shame they'll never meet.",
        "I'm reading a book about anti-gravity. It's impossible to put down.",
        "I used to play piano by ear, but now I use my hands.",
        "My boss told me to have a good day. So I went home.",
    ],
}

def joke_generator(category: str) -> str:
    category = category.strip()
    if category not in JOKES:
        return f"Unknown category {category!r}. Pick one of: {list(JOKES)}."
    return random.choice(JOKES[category])

# --- 2. The tool description we hand to the model -------------------------
tools = [{
    "type": "function",
    "function": {
        "name": "joke_generator",
        "description": "Return one random joke from a given category.",
        "parameters": {
            "type": "object",
            "properties": {
                "category": {
                    "type": "string",
                    "enum": ["AI", "programmer", "animals", "humans"],
                    "description": "Which category of joke to fetch.",
                },
            },
            "required": ["category"],
        },
    },
}]

# Map tool name -> Python callable. Lets us scale to many tools later.
TOOL_REGISTRY = {"joke_generator": joke_generator}

# --- 3. The agent loop: LLM + Memory + Tools ------------------------------
agent_history = [
    {"role": "system", "content": (
        "You are a cheerful comedian assistant. "
        "When the user asks for a joke, ALWAYS use the joke_generator tool — "
        "do not invent jokes yourself."
    )}
]

def agent_chat(user_message: str) -> str:
    agent_history.append({"role": "user", "content": user_message})

    while True:
        response = client.chat.completions.create(
            model=MODEL,
            messages=agent_history,
            tools=tools,
        )
        message = response.choices[0].message
        agent_history.append(message.model_dump(exclude_none=True))

        # No tool call -> the model answered with text. We're done.
        if not message.tool_calls:
            return message.content

        # Otherwise: run each requested tool and feed the result back.
        for tc in message.tool_calls:
            fn = TOOL_REGISTRY[tc.function.name]
            args = json.loads(tc.function.arguments)
            result = fn(**args)
            print(f"  [tool] {tc.function.name}({args}) -> {result}")
            agent_history.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result,
            })


print("=== FULL REQUEST ===")
print(json.dumps({
    "messages": agent_history,
    "tools": tools
}, indent=2))
print(" >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>><<<<<<<<<<<<<<<<<<<<<<<<<<<<<<")

# --- 4. Try it out --------------------------------------------------------
print("A1:", "Tell me a programmer joke, please.")
print("Q1:", agent_chat("Tell me a programmer joke, please."))
print("========================================================================")
print("Q2:", "Ok now tell me some usefull fact about programming?")
print("A2:", agent_chat("Ok now tell me some usefull fact about programming?"))
print("===========================================================================")
print("Q3:", "Nice! Now one joke about animals.")
print("A3:", agent_chat("Nice! Now one joke about animals."))


In [ ]:
print("==================================================")
print(json.dumps(agent_history, indent=2, ensure_ascii=False))

In [ ]:
import json

user_msg = "Tell me a programmer joke"

print("=== TOOLS GIVEN TO LLM ===")
print(json.dumps(tools, indent=2))

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Use tools when needed."},
        {"role": "user", "content": user_msg}
    ],
    tools=tools,
)

msg = response.choices[0].message

print("\n=== LLM RESPONSE ===")
print(msg)

if msg.tool_calls:
    print("\n=== TOOL CALL DETECTED ===")
    for tc in msg.tool_calls:
        print("Tool:", tc.function.name)
        print("Args:", tc.function.arguments)
else:
    print("\n=== NO TOOL USED ===")
    print("Answer:", msg.content)